In [1]:
import os
import pandas as pd


# 1. Mendapatkan path ke folder Desktop secara otomatis
folder_path = r'C:\protofolio\ecommerce_analytic\data mentah'

# 3. Membaca file menggunakan pandas
df_transactions = pd.read_csv(os.path.join(folder_path, 'transactions.csv')).copy()
df_transactions.head()

,transaction_id,timestamp,customer_id,product_id,quantity,discount_applied,gross_revenue,campaign_id,refund_flag
0,1,2021-12-27 08:25:15,59540,1630.0,3,0.00,43.74,0,0
1,2,2023-06-06 21:14:26,54871,1901.0,3,0.00,174.78,21,0
2,3,2023-08-31 05:29:54,51818,1884.0,1,0.00,40.61,37,0
3,4,2022-06-26 20:33:46,18164,1114.0,2,0.15,68.76,13,0
4,5,2023-07-26 18:12:35,86915,408.0,1,0.00,14.64,4,0


In [2]:
df_transactions.isnull().sum()

transaction_id          0
timestamp               0
customer_id             0
product_id          10449
quantity                0
discount_applied        0
gross_revenue       10449
campaign_id             0
refund_flag             0
dtype: int64

In [3]:
#df_transactions['refund_flag'].unique()
print(df_transactions['refund_flag'].value_counts())
#gross data minus
jumlah_minus = (df_transactions['gross_revenue'] < 0).sum()
print(f"Ada minus: {jumlah_minus}")


refund_flag
0    100098
1      3029
Name: count, dtype: int64
Ada minus: 2704


In [4]:
# Melihat sampel data yang gross_revenue-nya < 0
negative_revenue = df_transactions[df_transactions['gross_revenue'] < 0]

# berapa banyak dari data negatif ini yang punya refund_flag == 1
print(negative_revenue['refund_flag'].value_counts())

# gross_revenue negatif yang bukan refund (refund_flag == 0)
data_aneh = df_transactions[(df_transactions['gross_revenue'] < 0) & (df_transactions['refund_flag'] == 0)]

print(f"Jumlah data negatif yang BUKAN refund: {len(data_aneh)}")



refund_flag
1    2704
Name: count, dtype: int64
Jumlah data negatif yang BUKAN refund: 0


import numpy as np

df_transactions_clean = df_transactions.dropna(subset=['product_id', 'gross_revenue'])

# Membuat kolom baru bernama 'net_revenue'
df_transactions_clean['net_revenue'] = np.where(
    df_transactions_clean['refund_flag'] == 1, 
    0, 
    df_transactions_clean['gross_revenue']
)

# hasil
display(df_transactions_clean[['transaction_id', 'gross_revenue', 'refund_flag', 'net_revenue']].head(10))

In [5]:
df_transactions.dtypes

transaction_id        int64
timestamp               str
customer_id           int64
product_id          float64
quantity              int64
discount_applied    float64
gross_revenue       float64
campaign_id           int64
refund_flag           int64
dtype: object

In [6]:
# 3. create kolom transaction date

df_transactions['timestamp'] = pd.to_datetime(df_transactions['timestamp'])
df_transactions['transaction_date'] = df_transactions['timestamp'].dt.date

df_transactions.info()
#display(df_transactions_clean['transaction_date'])
#display(df_transactions_clean.head())

<class 'pandas.DataFrame'>
RangeIndex: 103127 entries, 0 to 103126
Data columns (total 10 columns):
 #   Column            Non-Null Count   Dtype         
---  ------            --------------   -----         
 0   transaction_id    103127 non-null  int64         
 1   timestamp         103127 non-null  datetime64[us]
 2   customer_id       103127 non-null  int64         
 3   product_id        92678 non-null   float64       
 4   quantity          103127 non-null  int64         
 5   discount_applied  103127 non-null  float64       
 6   gross_revenue     92678 non-null   float64       
 7   campaign_id       103127 non-null  int64         
 8   refund_flag       103127 non-null  int64         
 9   transaction_date  103127 non-null  object        
dtypes: datetime64[us](1), float64(3), int64(5), object(1)
memory usage: 7.9+ MB


In [7]:
print(df_transactions['refund_flag'].value_counts())

refund_flag
0    100098
1      3029
Name: count, dtype: int64


In [8]:
# Cek rentang tanggal transaksi
print(f"Transaksi dimulai dari: {df_transactions['transaction_date'].min()}")
print(f"Transaksi berakhir di: {df_transactions['transaction_date'].max()}")

Transaksi dimulai dari: 2021-01-01
Transaksi berakhir di: 2023-12-31


In [9]:
# save cleand data
df_transactions.to_csv(r'C:\protofolio\ecommerce_analytic\data bersih\dim_transactions_clean.csv', index=False)
print("File sudah tersimpan!")



File sudah tersimpan!
